In [20]:
!pip install sklearn-pandas pandas scikit-learn

In [21]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn_pandas import DataFrameMapper
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

URL_BASE = "https://raw.githubusercontent.com/FranciscoAmbort/grupo13_castellano_ambort_ia_2026/main/entrega_3/data"

url_train = f"{URL_BASE}/solicitudes_train.csv"
url_test = f"{URL_BASE}/solicitudes_test.csv"
url_comp = f"{URL_BASE}/comportamiento_bancario.csv"

# 2. Carga directa optimizada
df_sol_train = pd.read_csv(url_train)
df_sol_test = pd.read_csv(url_test)
comportamiento = pd.read_csv(url_comp)

In [22]:
df_compor_tratado = comportamiento.copy()
# Orden en base a la fecha
df_compor_tratado["Mes_Referencia"] = pd.to_datetime(df_compor_tratado["Mes_Referencia"])
df_compor_tratado = df_compor_tratado.sort_values(
    by=["ID_Usuario", "Mes_Referencia"], ascending=[True, True]
)

# ATraso de deuda maximo
df_atrasos = (
    df_compor_tratado.groupby("ID_Usuario")["Dias_Atraso_Otras_Deudas"]
    .agg(Dias_Atraso_Max="max", Dias_Atraso_Mean="mean")
    .reset_index()
)

# Promedio ultimos 6 meses
ultimos_6m = df_compor_tratado.groupby("ID_Usuario").tail(6)
df_saldo_reciente = (
    ultimos_6m.groupby("ID_Usuario")["Saldo_Promedio_Mensual"]
    .agg(Saldo_Prom_Ultimos_6M="mean")
    .reset_index()
)

# Agregamos columnas nuevas al comportamiento
comportamiento_procesado = pd.merge(
    df_atrasos, df_saldo_reciente, on="ID_Usuario", how="inner"
)

#Eliminamos la fecha
df_sol_train_tratado = df_sol_train.drop(columns=['Fecha_Solicitud']).copy()
df_sol_test_tratado = df_sol_test.drop(columns=['Fecha_Solicitud']).copy()

# Agregamos la nueva Feature Esfuerzo_Pago (Monto / Ingreso)
df_sol_train_tratado['Esfuerzo_Pago'] = (
    df_sol_train_tratado['Monto_Solicitado'] / (df_sol_train_tratado['Ingreso_Mensual'] + 1)
)
df_sol_test_tratado['Esfuerzo_Pago'] = (
    df_sol_test_tratado['Monto_Solicitado'] / (df_sol_test_tratado['Ingreso_Mensual'] + 1)
)

# Merge final hacia train_sol_comp_procesados
train_sol_comp_procesados = pd.merge(
    df_sol_train_tratado,
    comportamiento_procesado,
    on='ID_Usuario',
    how='left'
)
train_sol_comp_procesados['Tiene_Historial'] = (
    train_sol_comp_procesados['Saldo_Prom_Ultimos_6M'].notna().astype(int)
)

# Unimos con test
test_sol_comp_procesados = pd.merge(
    df_sol_test_tratado,
    comportamiento_procesado,
    on='ID_Usuario',
    how='left'
)
test_sol_comp_procesados['Tiene_Historial'] = (
    test_sol_comp_procesados['Saldo_Prom_Ultimos_6M'].notna().astype(int)
)

train_sol_comp_procesados.head()

train_procesado,validation_procesado= train_test_split(
    train_sol_comp_procesados,
    test_size=0.2,
    random_state=42,
    stratify=train_sol_comp_procesados['Estado_Final'] # para que haya en el conjunto de validación exactamente el mismo porcentaje de morosos y pagadores que en los datos de entrenamiento
)

In [23]:
# En este intento nos enfocamos en intentar entrenar el arbol Gradint Boosting.
# Como primer intento(submission_gradient_boosting.csv) utilizamos el data set original agregando Dias_Atraso_Max,Saldo_Prom_Ultimos_6M,Tiene_Historial,Esfuerzo_Pago, con esto obtuvimos un score bajo en kaggle(140.000). A partir de esto, decidimos
#modificar las features que le entregabamos con el objetivo de mejorar este score y evitar el sobreentrenamiento.

# Intento arbol Gradint Boosting

In [24]:
mapper_arbol = DataFrameMapper([
    (['Monto_Solicitado'], None),
    (['Tasa_Interes'], None),
    (['Ingreso_Mensual'], None),
    (['Motivo_Prestamo'], [OneHotEncoder()]),
    (['Esfuerzo_Pago'], None),
    (['Dias_Atraso_Max'], None),
    (['Dias_Atraso_Mean'], None),
    (['Saldo_Prom_Ultimos_6M'], None),
    (['Tiene_Historial'], None)
])
#al tratarse de arboles, no es necesario estandarizacion de las variables numericas.

#Pipeline
pipeline_gb = Pipeline([
    ('mapper', mapper_arbol),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100,        # Cantidad de arboles
        learning_rate=0.04,      # pasos del aprendizaje
        max_depth=2,             # profundidad del arbol
        min_samples_leaf=100,    # minimo para crear una nueva hoja
        subsample=0.8,           # porcentaje que utiliza para entrenar
        random_state=42
    ))
])

# 2. Entrenamiento
pipeline_gb.fit(
    train_procesado,
    train_procesado['Estado_Final']
)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Pipeline(steps=[('mapper',
                 DataFrameMapper(drop_cols=[],
                                 features=[(['Monto_Solicitado'], None),
                                           (['Tasa_Interes'], None),
                                           (['Ingreso_Mensual'], None),
                                           (['Motivo_Prestamo'],
                                            [OneHotEncoder()]),
                                           (['Esfuerzo_Pago'], None),
                                           (['Dias_Atraso_Max'], None),
                                           (['Dias_Atraso_Mean'], None),
                                           (['Saldo_Prom_Ultimos_6M'], None),
                                           (['Tiene_Historial'], None)])),
                ('classifier',
                 GradientBoostingClassifier(learning_rate=0.04, max_depth=2,
                                            min_samples_leaf=100,
                                            random_state=42, subsample=0.8))])

In [25]:
# probabilidades en test
test_eval_gb = test_sol_comp_procesados.copy()
test_eval_gb['prob_paga'] = pipeline_gb.predict_proba(test_sol_comp_procesados)[:, 1]

# 2. Ordenamiento: probabilidad descendente, en caso de empate, desempate con atraso y saldo
candidatos_gb = test_eval_gb[test_eval_gb['prob_paga'] >= 0.5].sort_values(
    by=['prob_paga', 'Dias_Atraso_Max', 'Saldo_Prom_Ultimos_6M'],
    ascending=[False, True, False]
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_gb = []

for idx, fila in candidatos_gb.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_gb.append(fila['ID_Solicitud'])
    else:
        continue

test_eval_gb['Decision_Aprobacion'] = test_eval_gb['ID_Solicitud'].isin(ids_aprobados_gb).astype(int)
submission_gb = test_eval_gb[['ID_Solicitud', 'Decision_Aprobacion']]
submission_gb.to_csv('submission_gradient_boosting.csv', index=False)

print(f"Total registros: {len(submission_gb)}")
print(f"Créditos aprobados: {submission_gb['Decision_Aprobacion'].sum()}")
print(f"Monto total colocado: ${monto_acumulado:,.2f} USD")

Total registros: 3061
Créditos aprobados: 839
Monto total colocado: $1,999,700.00 USD


In [26]:
#Aqui intentamos mejorar las features y utilizamos gradSearch para encontrar los hiperparametros optimos.

In [27]:
# copiamos
train_mejorado_all = train_sol_comp_procesados.copy()

#eliminamos las columnas innecesarias, ya que por ejemplo Tiene_Historial todos los valores eran ==1.
columnas_a_borrar = [
    'Dias_Atraso_Max',
    'Monto_Solicitado',
    'Tasa_Interes',
    'Tiene_Historial',
]
train_mejorado_all = train_mejorado_all.drop(columns=columnas_a_borrar)

# Agregamos la feature Uso_Dinero
train_mejorado_all['Uso_Dinero'] = train_mejorado_all['Saldo_Prom_Ultimos_6M'] / (
    train_mejorado_all['Ingreso_Mensual']
)

# Como estas columnas se utilizan en uso_dinero las eliminamos para evitar informacion duplicada y sumarle complejidad a la entrada
train_mejorado_all = train_mejorado_all.drop(
    columns=['Ingreso_Mensual', 'Saldo_Prom_Ultimos_6M']
)

# mapper
mapper_gb_9 = DataFrameMapper([
    (['Motivo_Prestamo'], [OneHotEncoder()]),
    (['Esfuerzo_Pago'],None),
    (['Dias_Atraso_Mean'],None),
    (['Uso_Dinero'],None),
])

gb_intento_9 = Pipeline([
    ('mapper', mapper_gb_9),
    (
        'classifier',
        GradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=3,
            min_samples_leaf=20,
            n_estimators=200,
            subsample=1.0,
            random_state=42,
        ),
    ),
])

# entrenamiento en train
gb_intento_9.fit(train_mejorado_all, train_mejorado_all['Estado_Final'])

# agregamos las nuevas features a test
test_eval_gb = test_sol_comp_procesados.copy()

test_eval_gb['Uso_Dinero'] = test_eval_gb['Saldo_Prom_Ultimos_6M'] / (
    test_eval_gb['Ingreso_Mensual'] + 1
)

test_eval_gb['prob_paga'] = gb_intento_9.predict_proba(test_eval_gb)[:, 1]

# ==============================================================================
# ordenamos en base a la probabilidad de pago
# ==============================================================================
candidatos_gb = test_eval_gb[test_eval_gb['prob_paga'] >= 0.5].sort_values(
    by='prob_paga', ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_gb = []

for idx, fila in candidatos_gb.iterrows():
  monto = fila['Monto_Solicitado']
  if monto_acumulado + monto <= PRESUPUESTO_MAX:
    monto_acumulado += monto
    ids_aprobados_gb.append(fila['ID_Solicitud'])
  else:
    continue

test_eval_gb['Decision_Aprobacion'] = (
    test_eval_gb['ID_Solicitud'].isin(ids_aprobados_gb).astype(int)
)
submission_gb = test_eval_gb[['ID_Solicitud', 'Decision_Aprobacion']]
submission_gb.to_csv('submission_gb_data_set_mejorado_corregido.csv', index=False) #esta version es la corregida debido a que en submission_gb_data_set_mejora entregado en Kaggle tambien, utilizamos Standarizacion para las variables numericas, lo cual no es necesario.}
#al corregir esto, la solucion mejoro unicamente 400 usd, con lo cual, obtenemos la conclusion de que estandarizar las variables en arboles no es necesario.

print(f"Total registros evaluados: {len(submission_gb)}")
print(f"Créditos aprobados:        {submission_gb['Decision_Aprobacion'].sum()}")
print(f"Monto total colocado:      ${monto_acumulado:,.2f} USD")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Total registros evaluados: 3061
Créditos aprobados:        892
Monto total colocado:      $1,999,600.00 USD
